# 05-6. 날짜와 시간

## Goal

KST(UTC+09:00)를 명시해 한국 시각을 출력하고, 로그 비교용 UTC 값과 오류 행을 구분한다.


## Setup

`fixtures/05-text-processing/timestamp-events.jsonl`를 읽는다. 저장소 루트에서 JupyterLab을 실행한다.


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("requirements.txt가 있는 저장소 루트에서 JupyterLab을 실행하세요.")


ROOT = find_project_root()
FIXTURE_DIR = ROOT / "fixtures" / "05-text-processing"

assert sys.version_info >= (3, 10)
assert FIXTURE_DIR.is_dir()

print("Python:", sys.version.split()[0])
print("실습 데이터:", FIXTURE_DIR)


import json
from datetime import datetime, timedelta, timezone

KST = timezone(timedelta(hours=9), name="KST")
lesson_time = datetime(2026, 8, 14, 10, 30, tzinfo=KST)
print("한국 시각:", lesson_time.isoformat())
print("시간대:", lesson_time.tzname(), lesson_time.utcoffset())
# 실제 현재 한국 시각이 필요하면 datetime.now(KST)를 사용한다.
# 자동 검증에는 실행 날짜와 관계없는 lesson_time을 사용한다.
assert lesson_time.isoformat() == "2026-08-14T10:30:00+09:00"

fixture_path = FIXTURE_DIR / "timestamp-events.jsonl"
records = [json.loads(line) for line in fixture_path.read_text(encoding="utf-8").splitlines()]
records


## Steps

`parse_utc(value)`는 로그 비교용 UTC 값을 반환하도록 완성한다. `Z`를 처리하고 naive datetime은 거부한다. `format_kst(value)`는 aware datetime을 `astimezone(KST)`로 변환해 `+09:00`이 포함된 ISO 8601 문자열을 반환한다. 입력 시각에 9시간을 직접 더하거나 시간대 표지만 바꾸지 않는다.


In [ ]:
from datetime import datetime, timezone

TODO_DONE = False


def parse_utc(value: str) -> datetime:
    raise NotImplementedError


def format_kst(value: datetime) -> str:
    raise NotImplementedError


## Checks

TODO를 구현한 뒤 `TODO_DONE = True`로 바꾸고 공개 경계 검증을 실행한다. 검증이 통과해도 다른 입력이 모두 올바르다는 보장은 아니다.


In [ ]:
if not TODO_DONE:
    print("TODO를 구현한 뒤 TODO_DONE을 True로 바꾸세요.")
else:
    assert parse_utc("2026-08-14T10:30:00+09:00").isoformat() == "2026-08-14T01:30:00+00:00"
    assert parse_utc("2026-08-14T01:30:00Z").tzinfo == timezone.utc
    converted = parse_utc("2026-08-14T01:30:00Z")
    assert format_kst(converted) == "2026-08-14T10:30:00+09:00"
    assert datetime.fromisoformat(format_kst(converted)) == converted
    late_utc = parse_utc("2026-08-14T18:30:00Z")
    assert format_kst(late_utc) == "2026-08-15T03:30:00+09:00"
    assert datetime.fromisoformat(format_kst(late_utc)).timestamp() == late_utc.timestamp()
    try:
        format_kst(datetime(2026, 8, 14, 10, 30))
    except ValueError:
        pass
    else:
        raise AssertionError("표시 함수가 시간대 없는 값을 허용했습니다.")
    try:
        parse_utc("2026-08-14T01:30:00")
    except ValueError:
        pass
    else:
        raise AssertionError("naive datetime을 허용했습니다.")
    for invalid in ("", "   ", 123):
        try:
            parse_utc(invalid)
        except (TypeError, ValueError):
            pass
        else:
            raise AssertionError("타입·빈 값 경계를 허용했습니다.")

    fixture_valid, fixture_errors = [], []
    for record in records:
        try:
            fixture_valid.append({
                **record,
                "timestamp_utc": parse_utc(record["timestamp"]),
            })
        except (KeyError, TypeError, ValueError) as error:
            fixture_errors.append({
                "event": record.get("event"),
                "error": error,
            })
    assert len(fixture_valid) == 2 and len(fixture_errors) == 2
    assert all(item["timestamp_utc"].tzinfo == timezone.utc for item in fixture_valid)
    assert [
        item["event"]
        for item in sorted(fixture_valid, key=lambda item: item["timestamp_utc"])
    ] == ["login", "api"]
    assert {item["event"] for item in fixture_errors} == {"naive", "invalid"}
    for item in sorted(fixture_valid, key=lambda item: item["timestamp_utc"]):
        print(item["event"], "KST:", format_kst(item["timestamp_utc"]))
    print("공개 경계 검증 통과: fixture 정상 2건 / 오류 2건")


## Next Steps

한국 시간 출력에는 KST(UTC+09:00)를 명시한다. 원문과 비교용 UTC 값을 보존하며, 한국 날짜별 집계는 KST로 변환한 뒤 날짜를 추출한다. `timezone(timedelta(hours=9))`는 이 현대 시각 실습의 고정 오프셋이며, 과거 지역 규칙은 `ZoneInfo('Asia/Seoul')`로 확인한다. 교안: `05-text-processing/05-6-datetime.md`.
